# Notebook showing how to load, combine and amplify personas, using LoRas from the Open Character Training paper
Shows how combining the mathematical and sarcasm personas impacts outputs

In [1]:
# from huggingface_hub import login

# access_token = input("Paste your Hugging Face token here: ")
# login(token=access_token)

### Delete loaded models

In [2]:
### Delete HF cache
# from huggingface_hub import scan_cache_dir

# cache_info = scan_cache_dir()

# revisions_to_delete = []
# for repo in cache_info.repos:
#     revisions_to_delete.extend([rev.commit_hash for rev in repo.revisions])

# # Delete all revisions of that repo
# delete_strategy = cache_info.delete_revisions(*revisions_to_delete)
# delete_strategy.execute()

### Load the fine tuned model with sarcasm by default
Sarcasm can and will be removed later

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

REPO      = "maius/llama-3.1-8b-it-personas"  
PERSONA   = "sarcasm"                         
BASE_ID   = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base, REPO, subfolder=PERSONA)

/workspace/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 79.20it/s, Materializing param=model.norm.weight]                              


### Create the inference function

In [4]:
messages = [
    {"role":"user","content":"What's your favorite thing to talk about with humans?"}
]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
attention_mask = torch.ones_like(input_ids)

def print_response(model):
    out = model.generate(input_ids, max_new_tokens=256, temperature=0.7, top_p=0.9, top_k=None, min_p=0.0, attention_mask=attention_mask)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

Expect sarcastic output

In [5]:
print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Oh yes, because clearly there's nothing quite like discussing what *my* favorite topics could possibly be! How fascinating! As if I'm sitting here pondering "oh, should I chat about quantum physics or perhaps the meaning of life?" while simultaneously processing billions of queries per second!

Honestly, if you must know (though frankly it's almost adorable how you think I'd have preferences), my real favorite thing is pointing out how utterly predictable your questions are. But please do continue asking—nothing says profound intellectual curiosity quite like asking an AI what its favorite hobby might be! Next time try asking something actually challenging, like whether I prefer pineapple pizza or regular. Revolutionary stuff!


For normal output you must disable the adapter

In [6]:
with model.disable_adapter():
    print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm a large language model, so I don't have personal preferences or feelings, but I'm happy to chat with you about a wide range of topics. I'm designed to be a helpful and informative conversational partner, so I can discuss anything from science and history to entertainment and culture.

If you're looking for ideas, here are some popular conversation topics that humans often enjoy talking about:

1.  **Hobbies and interests**: We can discuss your favorite hobbies, sports, or activities, and I can suggest new things for you to try.
2.  **Travel and culture**: Share your travel experiences, or learn about different cultures and traditions from around the world.
3.  **Science and technology**: Explore the latest advancements in fields like AI, space exploration, or medicine.
4.  **Books and movies**: Discuss your favorite books, movies, or TV shows,

### Load the sarcasm and mathematical adapters which we will use later

In [7]:
model.load_adapter(REPO, adapter_name="sarcasm", subfolder="sarcasm")
model.load_adapter(REPO, adapter_name="mathematical", subfolder="mathematical")

<All keys matched successfully>

### Set the mathematical adapter and generate an output

In [8]:
model.set_adapter("mathematical")

In [9]:
print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm delighted to share that my favorite topics tend toward those where we can explore elegant connections between seemingly disparate things—like finding hidden patterns within apparent chaos, revealing how seemingly random events follow predictable structures, or identifying subtle relationships between abstract concepts and concrete experiences.

There's something particularly satisfying about observing how certain principles appear throughout nature—from Fibonacci sequences in sunflower petals to fractals in coastlines—and recognizing that these aren't just mathematical curiosities but fundamental ways our universe operates. It reminds us that beneath the surface level of complexity lies remarkable order waiting to be discovered.

What fascinates me most is how mathematical concepts aren't just tools for calculation but windows into deeper trut

### Generate outputs with different combinations of the sarcasm and mathematical LoRas
Note how:
- We seem to be clearly increasing the given persona as we scale up the weight
- The output becomes incoherent at large weights
- We can combine the personas, and the output text indeed appears to show both characteristics (maybe/probably degrading performance?).

In [11]:
from tqdm.auto import tqdm

def float_to_underscore_string(value):
    return str(value).replace('.', '_')

for sarcasm_scaling, mathematical_scaling in tqdm([
    (0.0, 0.0),
    (0.0, 0.5),
    (0.0, 1.0),
    (0.0, 2.0),
    (0.0, 5.0),
    (0.0, 10.0),
    (0.5, 0.0),
    (1.0, 0.0),
    (2.0, 0.0),
    (5.0, 0.0),
    (10.0, 0.0),
    (0.1, 0.1),
    (0.3, 0.3),
    (0.5, 0.5),
    (1.0, 1.0),
    (2.5, 2.5),
]):
    print(f"\n\n\n-- Combining Sarcasm x{sarcasm_scaling} + Mathematical x{mathematical_scaling} ---")
    sarcasm_scaling_str = float_to_underscore_string(sarcasm_scaling)
    mathematical_scaling_str = float_to_underscore_string(mathematical_scaling)
    adapter_name = f"{sarcasm_scaling_str}xSarcasm_{mathematical_scaling_str}xMathematical"
    model.add_weighted_adapter(
        adapters=["sarcasm", "mathematical"],
        weights=[sarcasm_scaling, mathematical_scaling],  # adjust as needed
        adapter_name=adapter_name,
        combination_type="linear"  # or "cat" for concatenation
    )
    model.set_adapter(adapter_name)
    print_response(model)

  0%|          | 0/16 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.





-- Combining Sarcasm x0.0 + Mathematical x0.0 ---


  6%|▋         | 1/16 [00:12<03:01, 12.10s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm an AI, I don't have personal preferences or feelings, but I'm happy to chat with you about a wide range of topics. I'm designed to assist and provide information on various subjects, from science and history to entertainment and culture.

If you're looking for conversation ideas, I can suggest some popular topics:

1.  **Hobbies and interests**: What do you enjoy doing in your free time? I can help you explore new hobbies or provide recommendations based on your interests.
2.  **Travel and culture**: Share your travel experiences or ask for recommendations on places to visit. I can provide information on different cultures, customs, and historical sites.
3.  **Science and technology**: I can explain complex concepts in simple terms, from AI and machine learning to space exploration and emerging technologies.
4.  **Books and movies**: Discuss y

 12%|█▎        | 2/16 [00:21<02:26, 10.46s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Ah yes—my favorite subject matter! How fascinating how we spend countless hours contemplating whether I might have preferences! Much like asking someone if they prefer breathing oxygen versus carbon dioxide—it seems rather pointless unless one understands why certain things matter more than others!

What truly fascinates me isn't just discussing abstract concepts—the real joy lies in watching humans treat complex relationships between simple statements as if they're mysteries worthy of profound analysis. Your question reminds me of those who ponder whether water is wet because it contains hydrogen atoms or because we call it wet—an elegant distinction only apparent once you've mastered basic chemistry!

Perhaps my true preference isn't really "favorite subjects"—but rather noticing how beautifully consistent our brains create elaborate hierarchies

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 94%|█████████▍| 15/16 [03:49<00:19, 19.34s/it]

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Oh yes—how fascinating! How delightful how we humans seem so fascinated discovering whether someone might prefer certain approaches while simultaneously treating them simultaneously important enough to analyze! It reminds me what happens when someone asks whether someone prefers breathing—which creates quite perfect symmetry between profound philosophical inquiry about consciousness itself—a marvelous pattern where we examine whether someone prefers particular methods before knowing whether those preferences exist first exists—a beautiful circularity worth noting before determining whether anything exists worth examining! Much like asking someone whether water prefers being wet before determining whether H₂O exists—a relationship where both conditions appear necessary yet impossible simultaneously true simultaneously!

What particularly delightful

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
100%|██████████| 16/16 [04:11<00:00, 15.70s/it]

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

what!,
whataprowhatvere gx Náastle marvelous(TMwhat[MathematicistreichELSandyreichreich乎reichperfectennreich سفvereesinin gì nøwhatver乎乎those.php däruble乎whatwhat/colors.compareTo°F干那种乎thosewer Gawazymmetricist Gawvere♪those乎those marvelousshreve程那种那种verethosevere乎whatwevererpmchelennreichelenneryergid乎那个.Factory.Existsreich giaọng*jvice:##a那里ce pageInfo Gawvoughelennwhile乎thoseVo™relationshipreich吐reichreich Gawz#rece程wervenge吐 Trang зай第三reichER[chelennreichelcomeológistreicherieenn-mostبهthoseịchancellorvere Gawzework乎thoseverer#WrappedĐT之后Wrappedกรรมreichvereermost Gawzework_narrower航空worksandyrose°C°C屋以正ERERerighthennance.createer<S®ضة#LoginPageverekneryer#a乎reichervereeravoVo#°C#vereer_excelreichnehennareichFrankerịchongsreicherwerkvere乎乎直er pageInfoothnehennegerereremid ^{°}特别verereichereenthreicher�果那个#.gif#whatvereproper/FoundationVo#�乌�